In [1]:
import kagglehub
import pandas as pd
import os

In [2]:
# Preuzimanje dataseta
path = kagglehub.dataset_download(
    "retailrocket/ecommerce-dataset"
)

print("Dataset path:", path)

100%|██████████| 291M/291M [00:14<00:00, 21.1MB/s] 

Extracting files...


Dataset path: C:\Users\Milos\.cache\kagglehub\datasets\retailrocket\ecommerce-dataset\versions\2


In [4]:
# Prikaz fajlova
print(os.listdir(path))
# Učitavanje events.csv
events = pd.read_csv(
    os.path.join(path, "events.csv")
)

['category_tree.csv', 'events.csv', 'item_properties_part1.csv', 'item_properties_part2.csv']


In [5]:
# Osnovne informacije
print(events.head())
print(events.info())

print("\nBroj događaja:", len(events))
print("Broj korisnika:", events["visitorid"].nunique())
print("Broj proizvoda:", events["itemid"].nunique())

print("\nTipovi događaja:")
print(events["event"].value_counts())

       timestamp  visitorid event  itemid  transactionid
0  1433221332117     257597  view  355908            NaN
1  1433224214164     992329  view  248676            NaN
2  1433221999827     111016  view  318965            NaN
3  1433221955914     483717  view  253185            NaN
4  1433221337106     951259  view  367447            NaN
<class 'pandas.DataFrame'>
RangeIndex: 2756101 entries, 0 to 2756100
Data columns (total 5 columns):
 #   Column         Dtype  
---  ------         -----  
 0   timestamp      int64  
 1   visitorid      int64  
 2   event          str    
 3   itemid         int64  
 4   transactionid  float64
dtypes: float64(1), int64(3), str(1)
memory usage: 105.1 MB
None

Broj događaja: 2756101
Broj korisnika: 1407580
Broj proizvoda: 235061

Tipovi događaja:
event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64


In [6]:
score_map = {
    "view": 1,
    "addtocart": 3,
    "transaction": 5
}

events["score"] = events["event"].map(score_map)

print(events[["event", "score"]].head())

  event  score
0  view      1
1  view      1
2  view      1
3  view      1
4  view      1


In [15]:
# Broj interakcija po korisniku i proizvodu
user_counts = events["visitorid"].value_counts()
item_counts = events["itemid"].value_counts()

active_users = user_counts[user_counts >= 5].index
active_items = item_counts[item_counts >= 5].index

filtered = events[
    events["visitorid"].isin(active_users) &
    events["itemid"].isin(active_items)
]

print("Original shape:", events.shape)
print("Filtered shape:", filtered.shape)

print("Users:", filtered["visitorid"].nunique())
print("Items:", filtered["itemid"].nunique())

Original shape: (2756101, 6)
Filtered shape: (897028, 6)
Users: 81318
Items: 67625


In [16]:
filtered.to_csv(
    os.path.join(path, "filtered_events.csv"),
    index=False
)

In [17]:
# Broj interakcija po korisniku i proizvodu
user_counts = filtered["visitorid"].value_counts()
item_counts = filtered["itemid"].value_counts()

# Jači filter da matrica ne bude ogromna
active_users = user_counts[user_counts >= 20].index
popular_items = item_counts[item_counts >= 50].index

small = filtered[
    filtered["visitorid"].isin(active_users) &
    filtered["itemid"].isin(popular_items)
].copy()

print("Small shape:", small.shape)
print("Users:", small["visitorid"].nunique())
print("Items:", small["itemid"].nunique())

Small shape: (134812, 6)
Users: 5393
Items: 3422


In [18]:
top_items = (
    small["itemid"]
    .value_counts()
    .head(2000)
    .index
)

small = small[small["itemid"].isin(top_items)].copy()

print("Final shape:", small.shape)
print("Users:", small["visitorid"].nunique())
print("Items:", small["itemid"].nunique())

Final shape: (110286, 6)
Users: 5072
Items: 2000


In [19]:
user_item = small.pivot_table(
    index="visitorid",
    columns="itemid",
    values="score",
    aggfunc="max",
    fill_value=0
)

print(user_item.shape)
user_item.head()

(5072, 2000)


itemid,147,546,829,869,1152,1891,2416,2455,2463,2716,...,465522,465751,465951,466008,466064,466109,466135,466259,466385,466614
visitorid,,,,,,,,,,,,,,,,,,,,,
75,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
172,0,0,0,0,0,0,0,0,0,0,...,5,0,0,0,0,0,0,0,0,0
772,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1722,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1879,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [20]:
small.to_csv(
    os.path.join(path, "small_filtered_events.csv"),
    index=False
)

In [21]:
print("Final shape:", small.shape)
print("Users:", small["visitorid"].nunique())
print("Items:", small["itemid"].nunique())

Final shape: (110286, 6)
Users: 5072
Items: 2000


In [22]:
import numpy as np

X = user_item.values.astype(np.float32)

print(X.shape)
print(X.dtype)

(5072, 2000)
float32


In [23]:
np.save(
    os.path.join(path, "user_item_matrix.npy"),
    X
)

In [24]:
user_ids = user_item.index.to_numpy()
item_ids = user_item.columns.to_numpy()

np.save(os.path.join(path, "user_ids.npy"), user_ids)
np.save(os.path.join(path, "item_ids.npy"), item_ids)

In [25]:
X_train = X.copy()
test_items = {}

for user_idx in range(X.shape[0]):
    
    interacted = np.where(X[user_idx] > 0)[0]

    if len(interacted) < 5:
        continue

    n_test = max(1, int(len(interacted) * 0.2))

    hidden = np.random.choice(
        interacted,
        size=n_test,
        replace=False
    )

    X_train[user_idx, hidden] = 0

    test_items[user_idx] = hidden

print("Users with test items:", len(test_items))

Users with test items: 2138


In [27]:
import torch

X_train_tensor = torch.FloatTensor(X_train)
X_tensor = torch.FloatTensor(X)

print(X_train_tensor.shape)

torch.Size([5072, 2000])


In [30]:
# Popularity baseline: proizvodi sa najvećim ukupnim skorom
popular_items = (
    small.groupby("itemid")["score"]
    .sum()
    .sort_values(ascending=False)
)

print(popular_items.head(10))

itemid
119736    1131
461686     888
9877       748
312728     583
325310     486
241555     460
40870      414
38965      414
320130     410
29196      386
Name: score, dtype: int64


In [31]:
item_id_to_col = {
    item_id: idx
    for idx, item_id in enumerate(user_item.columns)
}

popular_cols = [
    item_id_to_col[item_id]
    for item_id in popular_items.index
    if item_id in item_id_to_col
]

print(len(popular_cols))

2000


In [32]:
def recall_at_k(recommended, relevant, k=10):
    recommended_k = recommended[:k]
    return len(set(recommended_k) & set(relevant)) / len(relevant)


def precision_at_k(recommended, relevant, k=10):
    recommended_k = recommended[:k]
    return len(set(recommended_k) & set(relevant)) / k

In [33]:
recalls = []
precisions = []

for user_idx, relevant_items in test_items.items():
    already_seen = set(np.where(X_train[user_idx] > 0)[0])

    recommendations = [
        item_col for item_col in popular_cols
        if item_col not in already_seen
    ]

    recalls.append(
        recall_at_k(recommendations, relevant_items, k=10)
    )

    precisions.append(
        precision_at_k(recommendations, relevant_items, k=10)
    )

print("Popularity Recall@10:", np.mean(recalls))
print("Popularity Precision@10:", np.mean(precisions))

Popularity Recall@10: 0.030268414014619827
Popularity Precision@10: 0.008746492048643594
